# CEC Quantum Computing — Lecture 7 practice

This practice notebook accompanies Lecture 7 on quantum machine learning.

Topics:
- image encoding for quantum circuits
- quantum kernels as overlaps of feature states
- hybrid quantum-classical training with PennyLane and PyTorch
- nonlinear expressivity via data re-uploading

Authors: Dr. Pavel Sulimov, Prof. Dr. R. M. Fuchslin, Claude Lehmann

Target stack:
- PennyLane >= 0.44
- PyTorch >= 2.5
- scikit-learn
- NumPy
- Matplotlib

All qubits start in $|0\rangle$ by default unless we explicitly prepare another state.


In [ ]:
# If needed:
# %pip install -q pennylane torch scikit-learn matplotlib

import math

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.datasets import load_digits, make_circles, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

np.random.seed(7)
torch.manual_seed(7)
torch.set_default_dtype(torch.float64)

print(f"PennyLane: {qml.__version__}")
print(f"Torch: {torch.__version__}")
print(f"NumPy: {np.__version__}")

## Part 1 — Image encoding in PennyLane

We start with a familiar classical object: a handwritten digit image. The goal is to understand what is gained and lost when we map image intensities into a quantum circuit.

We will use a pooled $4\times4$ version of an `sklearn` digits sample so that the encoding logic stays visible.


In [ ]:
digits = load_digits()
mask = np.isin(digits.target, [0, 1])
images = digits.images[mask] / 16.0
labels = digits.target[mask]

sample_image = images[labels == 1][0]


def average_pool_2x2(image: np.ndarray) -> np.ndarray:
    """Average-pool an 8x8 image into a 4x4 image."""
    return image.reshape(4, 2, 4, 2).mean(axis=(1, 3))


def quadrant_means(image_4x4: np.ndarray) -> np.ndarray:
    """Return four coarse features by averaging 2x2 quadrants."""
    return image_4x4.reshape(2, 2, 2, 2).mean(axis=(1, 3)).flatten()


pooled_image = average_pool_2x2(sample_image)
coarse_features = quadrant_means(pooled_image)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(sample_image, cmap="gray_r")
axes[0].set_title("Original 8x8 digit")
axes[0].axis("off")
axes[1].imshow(pooled_image, cmap="gray_r")
axes[1].set_title("Pooled 4x4 digit")
axes[1].axis("off")
plt.tight_layout()

print("coarse angle features:", np.round(coarse_features, 3))

### Task 1.1 — Compare angle and amplitude encoding

Use the pooled image to build two different encodings.

1. Use four quadrant averages as a shallow angle-encoding example on four qubits.
2. Use all sixteen pooled pixels in amplitude encoding on four qubits.
3. Compare the required qubit counts and the information that is visible after encoding.


In [ ]:
def qubits_for_angle(n_features: int) -> int:
    """Angle encoding uses one qubit per feature."""
    # YOUR CODE HERE
    raise NotImplementedError


def qubits_for_amplitude(n_features: int) -> int:
    """Amplitude encoding uses ceil(log2(n_features)) qubits."""
    # YOUR CODE HERE
    raise NotImplementedError


angle_dev = qml.device("default.qubit", wires=4)
amp_dev = qml.device("default.qubit", wires=4)


@qml.qnode(angle_dev)
def angle_encoder(features: np.ndarray):
    # YOUR CODE HERE
    raise NotImplementedError


@qml.qnode(amp_dev)
def amplitude_encoder(features: np.ndarray):
    # YOUR CODE HERE
    raise NotImplementedError


flat_pooled = pooled_image.flatten()
# YOUR CODE HERE: use coarse_features for angle encoding and flat_pooled for amplitude encoding
raise NotImplementedError

### Task 1.2 — Fixed qubit budget

Assume you have only four qubits. Compare how many raw image features can be represented by angle, dense-angle, and amplitude encoding.

This is not a statement about which method is always better. It only highlights the trade-off between qubit count and state-preparation complexity.


In [ ]:
encodings = ["angle", "dense-angle", "amplitude"]
# YOUR CODE HERE: fill the number of raw features that each encoding can represent with 4 qubits
features_per_4_qubits = ...

plt.figure(figsize=(6, 3.5))
plt.bar(encodings, features_per_4_qubits, color=["#00508C", "#007EA7", "#2E7D32"])
plt.ylabel("raw features represented")
plt.title("Capacity under a 4-qubit budget")
plt.tight_layout()

print("Interpret the plot in one sentence: which encoding uses qubits most densely?")

## Part 2 — Quantum kernel demo

We now make the kernel workflow explicit.

Given two data points $x$ and $y$, we build a feature map $U_\phi(x)$ and define

$$
K(x,y)=\left|\langle 0|^{\otimes n} U_\phi^\dagger(y) U_\phi(x) |0\rangle^{\otimes n}\right|^2.
$$

This is the probability of measuring the all-zero bitstring after:

1. uploading $x$ into the feature map,
2. applying the inverse feature map for $y$,
3. measuring the circuit.

Below we use a small two-qubit ZZ-style feature map so that the whole procedure stays transparent.


In [ ]:
X_circles, y_circles = make_circles(n_samples=90, noise=0.08, factor=0.35, random_state=7)
scaler_circles = MinMaxScaler(feature_range=(0.0, np.pi))
X_circles = scaler_circles.fit_transform(X_circles)
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_circles, y_circles, test_size=0.25, random_state=7, stratify=y_circles
)

plt.figure(figsize=(4.2, 4.0))
plt.scatter(X_train_k[:, 0], X_train_k[:, 1], c=y_train_k, cmap="coolwarm", edgecolor="k")
plt.title("Training data for the kernel demo")
plt.xlabel("x1")
plt.ylabel("x2")
plt.tight_layout()

kernel_dev = qml.device("default.qubit", wires=2)


def zz_feature_map(x: np.ndarray):
    """A tiny ZZ-style feature map with single-feature and pairwise terms."""
    # YOUR CODE HERE
    raise NotImplementedError


@qml.qnode(kernel_dev)
def feature_state(x: np.ndarray):
    # YOUR CODE HERE
    raise NotImplementedError


@qml.qnode(kernel_dev)
def overlap_probability(x: np.ndarray, y: np.ndarray):
    # YOUR CODE HERE: apply the feature map for x, then the adjoint map for y,
    # and return the probabilities on both qubits.
    raise NotImplementedError


def quantum_kernel(x: np.ndarray, y: np.ndarray) -> float:
    # YOUR CODE HERE: optionally compare the state-overlap view and the overlap-circuit view,
    # then return the probability of the all-zero state.
    raise NotImplementedError


def kernel_matrix(XA: np.ndarray, XB: np.ndarray) -> np.ndarray:
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE:
# 1. Evaluate one overlap circuit and print P(00).
# 2. Build K_train and K_test.
# 3. Fit a precomputed-kernel SVC (QSVM-style workflow).
# 4. Compare it with a linear SVC.
# 5. Plot both the train kernel matrix and the final accuracy comparison.
raise NotImplementedError

## Part 3 — Hybrid quantum-classical model with PyTorch

We now place a quantum layer inside a small PyTorch model. The dataset is intentionally tiny so that the whole training loop stays transparent.

The model uses a PennyLane `QNode` as a differentiable feature extractor and a classical dense head for the final prediction.


In [ ]:
X_moons, y_moons = make_moons(n_samples=220, noise=0.15, random_state=7)
moon_scaler = MinMaxScaler(feature_range=(-np.pi / 2, np.pi / 2))
X_moons = moon_scaler.fit_transform(X_moons)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=7, stratify=y_moons
)

X_train_t = torch.tensor(X_train_h)
X_test_t = torch.tensor(X_test_h)
y_train_t = torch.tensor(y_train_h, dtype=torch.float64)
y_test_t = torch.tensor(y_test_h, dtype=torch.float64)

hybrid_dev = qml.device("default.qubit", wires=2)


@qml.qnode(hybrid_dev, interface="torch")
def qlayer(inputs, weights):
    # YOUR CODE HERE
    raise NotImplementedError


weight_shapes = {"weights": (2, 2, 3)}


class HybridClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError


# YOUR CODE HERE: instantiate the model, train it with BCEWithLogitsLoss,
# report train/test accuracy, and plot the loss curve.
raise NotImplementedError

## Part 4 — Nonlinearity mini-lab via measurement and data re-uploading

We connect this section directly to the lecture theory.

1. First we verify the simplest example of effective nonlinearity:

$$
|\phi(x)\rangle = R_Y(x)|0\rangle, \qquad \langle Z \rangle = \cos(x).
$$

2. Then we compare one re-upload with three re-uploads on a target function that contains higher harmonics.

The goal is to see that repeated data injection increases expressive power without increasing the number of qubits.


In [ ]:
x_reg = torch.linspace(-math.pi, math.pi, 80)
y_reg = torch.sin(x_reg) + 0.35 * torch.sin(3 * x_reg)

one_qubit_dev = qml.device("default.qubit", wires=1)


@qml.qnode(one_qubit_dev)
def one_qubit_expectation(x):
    # YOUR CODE HERE: apply RY(x) and return <Z>
    raise NotImplementedError


# YOUR CODE HERE: evaluate the circuit on all x_reg points and compare it with np.cos(x_reg)
raise NotImplementedError


def make_reupload_model(num_reuploads: int) -> nn.Module:
    dev = qml.device("default.qubit", wires=1)

    @qml.qnode(dev, interface="torch")
    def circuit(x, weights):
        # YOUR CODE HERE
        raise NotImplementedError

    class ReuploadRegressor(nn.Module):
        def __init__(self):
            super().__init__()
            self.weights = nn.Parameter(0.1 * torch.randn(num_reuploads, 3))

        def forward(self, x_batch):
            # YOUR CODE HERE
            raise NotImplementedError

    return ReuploadRegressor()


def train_reupload(num_reuploads: int, epochs: int = 120, lr: float = 0.08):
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE: train models with 1 and 3 re-uploads and plot
# (1) the analytic cos(x) curve versus the measured <Z> curve,
# (2) the target function and both fitted curves,
# (3) both loss curves.
raise NotImplementedError

## References

- IBM Quantum Learning, *Introduction to Quantum Machine Learning*.
- IBM Quantum Learning, *Data encoding*.
- IBM Quantum Learning, *Quantum kernel methods*.
- IBM Quantum Learning, *QVCs and QNNs*.
- Maria Schuld and Francesco Petruccione, *Machine Learning with Quantum Computers*.
- Adrián Pérez-Salinas et al., *Data re-uploading for a universal quantum classifier*.
